### After batches processed, I want to retrieve the embeddings from client.file and save them in chromadb database
If I turn the computer off. Then I need to run the cells below

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
from openai import OpenAI
client = OpenAI()

In [7]:
batch_1_description =  'Content embeddings (antonio_m_lancuentra) 2025-11-11 14:33:19'
batch_2_description =  'Content embeddings (antonio_m_lancuentra) 2025-11-11 17:30:25'
batch_3_description =  'Content embeddings (antonio_m_lancuentra) 2025-11-12 17:42:22'

In [ ]:
batch_processes = client.batches.list().to_dict()
batch_info= [
    {'batch_id': batch['id'],
     'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'input_file_id': batch['input_file_id'],
    'output_file_id': batch['output_file_id']}  
            for batch in batch_processes['data'] if batch['metadata']['description'] in (batch_1_description, batch_2_description, batch_3_description)
    ]
batch_info

In [ ]:
batch_complete = [
    batch  for batch in batch_info if batch['status'] == 'completed'
]
batch_complete
# all 7 batches completed from 3 different batch processes

In [39]:
response = client.files.content(batch_complete[0]['output_file_id'])
text_response = response.text
lines = text_response.split('\n')
print(lines[0])
# There are 10001 lines per batch. 1 line, the embedding of 1 chunk

{"id": "batch_req_691512ff2140819094f6b433bcac53a8", "custom_id": "20000_65787_96142", "response": {"status_code": 200, "request_id": "bb092ea03de7fb51b757b5e7977d5f24", "body": {"object": "list", "data": [{"object": "embedding", "index": 0, "embedding": [0.0075749117, 0.026214456, 0.064089015, 0.00705907, -0.0001617415, -0.029634938, -0.020024356, -0.0011866093, -0.009015114, 0.01437433, 0.035728104, -0.032127596, 0.0007006274, -0.0017474573, 0.061319396, 0.028942533, -0.038913168, -0.0019958576, -0.013903494, 0.0452279, 0.002027016, 0.025702078, 0.038359243, -0.025522051, 0.0474436, 0.017199343, 0.010496861, -0.0011805507, 0.014748229, -0.0032214148, 0.016216127, -0.0029808038, -0.02071676, -0.0108222915, 0.008724304, 0.011036937, 0.020467494, 0.03808228, -0.03750066, -0.016506938, 0.022627799, -0.0038601584, -0.047194332, 0.040768813, 0.024178786, -0.0053765257, -0.0034931838, -0.04531099, -0.003673209, 0.0020910634, -0.01794714, 0.05866056, -0.009638279, 0.058328204, -0.016091494, 

In [40]:
import json 

def get_text_and_embeddings(batch):
    embedding_lines =  get_content_from_file(batch, 'output_file_id')
    text_lines = get_content_from_file(batch, 'input_file_id')
    return embedding_lines, text_lines

def get_content_from_file(batch, key):
    file = client.files.content(batch[key])
    text = file.text
    lines = text.split('\n')
    content_lines = [json.loads(line) for line in lines if line.strip()]
    return content_lines

In [41]:
def create_chroma_inputs(embedding_lines, text_lines):
    chroma_inputs = []
    text_dict = {item['custom_id']: item['body']['input'] for item in text_lines}
    for embed_item in embedding_lines:
        custom_id = embed_item['custom_id']
        text = text_dict.get(custom_id, "")
        chroma_input = {
            'id': embed_item['custom_id'],
            'embedding': embed_item['response']['body']['data'][0]['embedding'],
            'text': text
        }
        chroma_inputs.append(chroma_input)
    return chroma_inputs

In [42]:
from tqdm import tqdm

def process_batch_for_chromadb(batch):
    embedding_lines, text_lines = get_text_and_embeddings(batch)
    chroma_inputs = create_chroma_inputs(embedding_lines, text_lines)
    return chroma_inputs

def process_batches_for_chromadb(batches):
    all_chroma_inputs = []
    for batch in tqdm(batches, desc="Processing batches"):
        chroma_inputs = process_batch_for_chromadb(batch)
        all_chroma_inputs.extend(chroma_inputs)
    return all_chroma_inputs

In [43]:
chroma_inputs = process_batches_for_chromadb(batch_complete)

Processing batches: 100%|██████████| 7/7 [02:34<00:00, 22.01s/it]


In [ ]:
# chroma_inputs is a list with 63427 chunks
# every chunk is a dictionary with 3 items: id, embedding, text
# len(chroma_inputs[1]['embedding'])  # 1536 dimensions each embedding

In [ ]:
# Then save the file
with open('./chroma_inputs.jsonl', 'w') as f:
    for item in chroma_inputs:
        f.write(json.dumps(item) + '\n')